# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 6.9 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
TASK_ID = 'task279'
CH = 10
H = W = 30
LOCAL_TASK_JSON = Path('/mnt/data/task279.json')
KAGGLE_TASK_JSON = Path(COMPETITION) / f'{TASK_ID}.json'
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
OUT_DIR = Path.cwd() / f'{TASK_ID}_component_hole_recolor_onnx'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f'{TASK_ID}.onnx'
SUBMISSION_PATH = Path.cwd() / 'submission.zip'
SUMMARY_PATH = OUT_DIR / f'{TASK_ID}_validation_summary.json'
with TASK_JSON.open('r') as f:
    task=json.load(f)
print(TASK_ID, len(task.get('train', [])), len(task.get('test', [])), len(task.get('arc-gen', [])))

task279 4 1 261


In [6]:
def grid_to_tensor(grid, h=H, w=W, ch=CH, offset=(0,0)):
    x=np.zeros((1,ch,h,w),dtype=np.float32)
    ro,co=offset
    for r,row in enumerate(grid):
        for c,v in enumerate(row):
            rr,cc=r+ro,c+co
            if 0 <= rr < h and 0 <= cc < w:
                x[0,int(v),rr,cc]=1.0
    return x

def tensor_to_grid(y, h, w):
    return y[0,:,:h,:w].argmax(axis=0).astype(int).tolist()

def expected_tensor(ex):
    return grid_to_tensor(ex['output'])

def shifted_pair(ex, offset):
    return grid_to_tensor(ex['input'], offset=offset), grid_to_tensor(ex['output'], offset=offset)

In [7]:
class Task279ComponentHoleRecolor(nn.Module):
    """Recolor only foreground components that enclose a background hole.

    Structural rule:
    - active_canvas = sum(input_channels) > 0, so zero padding is not background color.
    - Treat color 9 inside active_canvas as canvas/background and color 1 as foreground strokes.
    - Static 4-neighbor flood-fill from active-canvas boundary through color-9 background marks outside-connected background.
    - Remaining color-9 pixels are enclosed holes.
    - Any color-1 component adjacent to a hole is selected by another static 4-neighbor flood-fill within color 1.
    - Selected color-1 pixels become color 8; all other pixels are preserved.
    - Output outside active_canvas is forced to all-zero across all channels.

    ONNX export is fixed [1,10,30,30] -> [1,10,30,30] and contains no Loop/Scan/NonZero/Unique/Script/Function.
    """
    def __init__(self, steps=60):
        super().__init__()
        self.steps=steps
        cross=np.array([[0,1,0],[1,1,1],[0,1,0]], dtype=np.float32).reshape(1,1,3,3)
        self.register_buffer('cross', torch.from_numpy(cross))
        edge=np.zeros((1,1,H,W), dtype=np.float32)
        edge[:,:,0,:]=1.0
        edge[:,:,-1,:]=1.0
        edge[:,:,:,0]=1.0
        edge[:,:,:,-1]=1.0
        self.register_buffer('tensor_edge', torch.from_numpy(edge))

    def expand4(self, z):
        return (F.conv2d(z, self.cross, padding=1) > 0.5).float()

    def forward(self, x):
        active=(x.sum(dim=1, keepdim=True) > 0.5).float()
        one=x[:,1:2] * active
        bg=x[:,9:10] * active

        # Boundary of the active canvas: tensor edge or adjacent to inactive padding.
        near_inactive=(F.conv2d(1.0-active, self.cross, padding=1) > 0.5).float()
        active_boundary=active * torch.clamp(self.tensor_edge + near_inactive, 0.0, 1.0)

        # Background reachable from outside is not a hole.
        reachable=bg * active_boundary
        for _ in range(self.steps):
            reachable = bg * self.expand4(reachable)
        hole = bg * (1.0 - reachable)

        # Seed the color-1 components touching a hole, then flood-fill inside color-1.
        seed = one * (F.conv2d(hole, self.cross, padding=1) > 0.5).float()
        selected = seed
        for _ in range(self.steps):
            selected = one * self.expand4(selected)

        outs=[]
        for ch in range(CH):
            if ch == 1:
                out = one * (1.0 - selected)
            elif ch == 8:
                out = torch.clamp(x[:,8:9] * active + selected, 0.0, 1.0)
            else:
                out = x[:,ch:ch+1] * active
            outs.append(out)
        return torch.cat(outs, dim=1) * active

model=Task279ComponentHoleRecolor().eval()


In [8]:
# Fast PyTorch sanity check before export.
with torch.no_grad():
    for split in ['train','test']:
        ok=0
        for ex in task.get(split, []):
            y=model(torch.from_numpy(grid_to_tensor(ex['input']))).numpy()
            ok += int(np.array_equal((y>0.5).astype(np.float32), expected_tensor(ex)))
        print(split, ok, '/', len(task.get(split, [])))

train 4 / 4
test 1 / 1


In [9]:
dummy = torch.from_numpy(grid_to_tensor(task['test'][0]['input']))
torch.onnx.export(
    model, dummy, str(ONNX_PATH), input_names=['input'], output_names=['output'],
    opset_version=17, do_constant_folding=True, dynamic_axes=None, dynamo=False,
)
onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))
print('ONNX:', ONNX_PATH, 'bytes:', ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/2058970932.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX: /kaggle/working/task279_component_hole_recolor_onnx/task279.onnx bytes: 96319


In [10]:
def vi_shape(vi):
    return [int(d.dim_value) if d.dim_value else (d.dim_param or None) for d in vi.type.tensor_type.shape.dim]
onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
summary_static = {
    'input_shape': vi_shape(onnx_model.graph.input[0]),
    'output_shape': vi_shape(onnx_model.graph.output[0]),
    'onnx_size_bytes': ONNX_PATH.stat().st_size,
    'ops': dict(ops),
    'forbidden_ops': sorted(forbidden & set(ops)),
}
print(json.dumps(summary_static, indent=2)[:4000])
assert summary_static['input_shape'] == [1,10,30,30]
assert summary_static['output_shape'] == [1,10,30,30]
assert summary_static['onnx_size_bytes'] < 1_400_000
assert not summary_static['forbidden_ops']

{
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 96319,
  "ops": {
    "Constant": 171,
    "ReduceSum": 1,
    "Greater": 123,
    "Cast": 123,
    "Slice": 10,
    "Mul": 136,
    "Sub": 3,
    "Conv": 122,
    "Add": 2,
    "Clip": 2,
    "Concat": 1
  },
  "forbidden_ops": []
}


In [11]:
sess_options = ort.SessionOptions(); sess_options.intra_op_num_threads=1; sess_options.inter_op_num_threads=1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=['CPUExecutionProvider'])

def validate_examples(examples):
    ok=0; bad=[]; outside_zero_ok=0; active_exact_ok=0
    for i,ex in enumerate(examples):
        x=grid_to_tensor(ex['input'])
        y=sess.run(None, {'input':x})[0]
        pred=(y>0.5).astype(np.float32)
        exp=expected_tensor(ex)
        if np.array_equal(pred, exp):
            ok += 1
        else:
            bad.append(i)
        active=(x.sum(axis=1, keepdims=True)>0.5).astype(np.float32)
        outside_zero_ok += bool(np.all(pred*(1-active)==0))
        active_exact_ok += bool(np.array_equal(pred*active, exp*active))
    return {'ok':ok, 'total':len(examples), 'bad_first10':bad[:10], 'outside_zero_ok':outside_zero_ok, 'active_exact_ok':active_exact_ok}

rng=random.Random(0)
inds=list(range(len(task.get('arc-gen', []))))
rng.shuffle(inds)
holdout=[task['arc-gen'][i] for i in inds[:max(1,int(0.60*len(inds)))]] if inds else []
rest=[task['arc-gen'][i] for i in inds[max(1,int(0.60*len(inds))):]] if inds else []
summary={
    'train': validate_examples(task.get('train', [])),
    'test': validate_examples(task.get('test', [])),
    'arc_gen_60_percent_holdout': validate_examples(holdout),
    'arc_gen_rest': validate_examples(rest),
    'arc_gen_full_diagnostic': validate_examples(task.get('arc-gen', [])),
}
print(json.dumps(summary, indent=2))
assert summary['train']['ok'] == summary['train']['total']
assert summary['test']['ok'] == summary['test']['total']
if holdout:
    assert summary['arc_gen_60_percent_holdout']['ok'] == summary['arc_gen_60_percent_holdout']['total']
assert summary['train']['outside_zero_ok'] == summary['train']['total']
assert summary['test']['outside_zero_ok'] == summary['test']['total']

{
  "train": {
    "ok": 4,
    "total": 4,
    "bad_first10": [],
    "outside_zero_ok": 4,
    "active_exact_ok": 4
  },
  "test": {
    "ok": 1,
    "total": 1,
    "bad_first10": [],
    "outside_zero_ok": 1,
    "active_exact_ok": 1
  },
  "arc_gen_60_percent_holdout": {
    "ok": 156,
    "total": 156,
    "bad_first10": [],
    "outside_zero_ok": 156,
    "active_exact_ok": 156
  },
  "arc_gen_rest": {
    "ok": 105,
    "total": 105,
    "bad_first10": [],
    "outside_zero_ok": 105,
    "active_exact_ok": 105
  },
  "arc_gen_full_diagnostic": {
    "ok": 261,
    "total": 261,
    "bad_first10": [],
    "outside_zero_ok": 261,
    "active_exact_ok": 261
  }
}


In [12]:
# Shifted active-canvas diagnostic: same rule should work when the visible grid is not at the tensor origin.
def validate_shifted(examples, offsets=((2,3),(5,7))):
    total=0; ok=0; outside_zero_ok=0
    for ex in examples:
        h=len(ex['input']); w=len(ex['input'][0])
        for off in offsets:
            if off[0]+h > H or off[1]+w > W:
                continue
            x, exp = shifted_pair(ex, off)
            y=sess.run(None, {'input':x})[0]
            pred=(y>0.5).astype(np.float32)
            active=(x.sum(axis=1, keepdims=True)>0.5).astype(np.float32)
            ok += bool(np.array_equal(pred, exp))
            outside_zero_ok += bool(np.all(pred*(1-active)==0))
            total += 1
    return {'ok':ok, 'total':total, 'outside_zero_ok':outside_zero_ok}

shifted_summary = validate_shifted(task.get('train', []) + task.get('test', []))
print(json.dumps({'shifted_canvas_diagnostic': shifted_summary}, indent=2))
assert shifted_summary['ok'] == shifted_summary['total']
assert shifted_summary['outside_zero_ok'] == shifted_summary['total']
summary['shifted_canvas_diagnostic'] = shifted_summary
summary['onnx_static'] = summary_static
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

{
  "shifted_canvas_diagnostic": {
    "ok": 10,
    "total": 10,
    "outside_zero_ok": 10
  }
}


1211

In [13]:
# Build Kaggle submission.zip with exactly one ONNX model file.
if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()
with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')
print('submission:', SUBMISSION_PATH, 'bytes:', SUBMISSION_PATH.stat().st_size)
print('contents:', zipfile.ZipFile(SUBMISSION_PATH).namelist())

submission: /kaggle/working/submission.zip bytes: 9910
contents: ['task279.onnx']
